In [1]:
import os
import pandas as pd
from google.cloud import bigquery

/opt/conda/envs/jupyter/lib/python3.12/site-packages/google/api_core/_python_package_support.py:206: FutureWarning: Package google.api_core depends on grpcio, currently installed at version 1.78.1. grpcio < 1.83.0 does not support Post-Quantum Cryptography (PQC). Support for non-PQC environments is deprecated. In April 2027, Google Cloud Python packages will raise their minimum requirements (including google-api-core, grpcio, and grpcio-status) to enforce grpcio >= 1.83.0. For more details on Google Cloud's post-quantum security migration, visit: https://cloud.google.com/security/resources/post-quantum-cryptography
  warnings.warn(
/opt/conda/envs/jupyter/lib/python3.12/site-packages/google/api_core/_python_package_support.py:206: FutureWarning: Package google.cloud.bigquery_storage_v1 depends on grpcio, currently installed at version 1.78.1. grpcio < 1.83.0 does not support Post-Quantum Cryptography (PQC). Support for non-PQC environments is deprecated. In April 2027, Google Cloud Pyt

In [10]:

dataset = "wb-silky-artichoke-2408.C2025Q4R6"
GOOGLE_PROJECT = "wb-shiny-lentil-3959"

In [15]:

manifest = pd.read_csv("/home/jupyter/workspace/Data from All of Us Controlled Tier /"+"vwb-aou-datasets-controlled-v9/v9/multiomics/rnaseq/manifest.tsv", sep="\t", dtype={"person_id": "string"})
# Assumes manifest.tsv contains only RNA-seq participants.
manifest = pd.read_csv("/home/jupyter/workspace/Data from All of Us Controlled Tier /"+"vwb-aou-datasets-controlled-v9/v9/multiomics/rnaseq/manifest.tsv", sep="\t", dtype={"sampleid": "string"})



# All of Us Researcher Workbench sets this environment variable.
dataset = os.environ["WORKSPACE_CDR"]
client = bigquery.Client()


In [16]:
person_ids = (
    pd.to_numeric(manifest["sampleid"], errors="raise")
    .dropna()
    .astype("int64")
    .drop_duplicates()
    .tolist()
)

In [36]:
query = f"""
WITH covid_root AS (
    SELECT concept_id
    FROM `{dataset}.concept`
    WHERE vocabulary_id = 'SNOMED'
      AND concept_code = '840539006'
),
covid_concepts AS (
    SELECT concept_id FROM covid_root
    UNION DISTINCT
    SELECT ca.descendant_concept_id
    FROM `{dataset}.concept_ancestor` AS ca
    JOIN covid_root AS root
      ON ca.ancestor_concept_id = root.concept_id
),
covid_diagnoses AS (
    SELECT DISTINCT
        co.person_id,
        co.condition_start_date AS covid_diagnosis_date
    FROM `{dataset}.condition_occurrence` AS co
    JOIN covid_concepts AS covid
      ON co.condition_concept_id = covid.concept_id
    WHERE co.person_id IN UNNEST(@person_ids)
),
blood_root AS (
    SELECT concept_id
    FROM `{dataset}.concept`
    WHERE vocabulary_id = 'SNOMED'
      AND concept_code = '119297000'  -- Blood specimen
),
blood_concepts AS (
    SELECT concept_id FROM blood_root
    UNION DISTINCT
    SELECT ca.descendant_concept_id
    FROM `{dataset}.concept_ancestor` AS ca
    JOIN blood_root AS root
      ON ca.ancestor_concept_id = root.concept_id
),
blood_specimens AS (
    SELECT
        s.person_id,
        s.specimen_id,
        s.specimen_date AS blood_specimen_date,
        s.specimen_datetime AS blood_specimen_datetime,
        c.concept_name AS specimen_concept_name
    FROM `{dataset}.specimen` AS s
    JOIN blood_concepts AS blood
      ON s.specimen_concept_id = blood.concept_id
    LEFT JOIN `{dataset}.concept` AS c
      ON s.specimen_concept_id = c.concept_id
    WHERE s.person_id IN UNNEST(@person_ids)
)
SELECT
    covid.person_id,
    TRUE AS has_covid_diagnosis,
    covid.covid_diagnosis_date,
    blood.specimen_id,
    blood.blood_specimen_date,
    blood.blood_specimen_datetime,
    blood.specimen_concept_name
FROM covid_diagnoses AS covid
LEFT JOIN blood_specimens AS blood
  ON covid.person_id = blood.person_id
ORDER BY
    covid.person_id,
    covid.covid_diagnosis_date,
    blood.blood_specimen_date,
    blood.blood_specimen_datetime,
    blood.specimen_id
"""

job_config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ArrayQueryParameter("person_ids", "INT64", person_ids)
    ]
)

results = client.query(query, job_config=job_config).to_dataframe()
display(results.head())

results.to_csv("rna_seq_covid_blood_specimen_dates.csv", index=False)